# 06 — HikingTTE-like LSTM + GIS features

Расширение подхода Asako et al. (HikingTTE) семантическими GIS-признаками:
on_trail, on_road, near_water, custom_difficulty, log_dist_trail, log_dist_road.

Авторы HikingTTE используют только GPX + slope + индивидуальную способность.
Наш вклад: добавляем признаки проходимости из OSM — то, что авторы явно называют ограничением.

**Постановка задачи (адаптация к сегментам):**
Вход: последовательность сегментов трека [slope, dist, elev_diff, GIS-фичи, ...].
Выход: скорость каждого сегмента (local head) + суммарное время остатка (entire head).
Для cost-surface SAR-симуляции используем только local head → per-cell speed.

In [ ]:
import copy, math, pickle, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')

CACHE_DIR  = Path('cache')
MODELS_DIR = Path('models')
MODELS_DIR.mkdir(exist_ok=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'device: {device}')

## 1. Загрузка данных

In [ ]:
# Базовый датасет сегментов (из notebook 03)
df_raw = pickle.load(open(CACHE_DIR / 'hikr_model_df.pkl', 'rb'))

# OSM-метки (из notebook 05, только Альпы)
osm_labels = pickle.load(open(CACHE_DIR / 'hikr_osm_labels.pkl', 'rb'))

# Фильтр: только Альпы (там есть OSM-метки)
alps_mask = (
    (df_raw['lat_mid'] >= 43) & (df_raw['lat_mid'] <= 49) &
    (df_raw['lon_mid'] >= 5)  & (df_raw['lon_mid'] <= 20)
)
df = df_raw[alps_mask].copy().reset_index(drop=True)

# Добавляем OSM-фичи
df['dist_trail_m']    = osm_labels['dist_trail_m']
df['dist_road_m']     = osm_labels['dist_road_m']
df['dist_water_m']    = osm_labels['dist_water_m']
# int чтобы CatBoost принимал как категориальные (NB-05 тоже использует int)
df['on_trail']        = (df['dist_trail_m'] < 50).astype(int)
df['on_road']         = (df['dist_road_m']  < 30).astype(int)
df['near_water']      = (df['dist_water_m'] < 100).astype(int)
df['off_trail']       = ((df['on_trail'] == 0) & (df['on_road'] == 0)).astype(int)
df['log_dist_trail']  = np.log1p(df['dist_trail_m'].clip(0, 5000))
df['log_dist_road']   = np.log1p(df['dist_road_m'].clip(0, 5000))

# Кастомная сложность (как в notebook 05)
slope = df['slope_deg'].values
road  = df['on_road'].values.astype(bool)
trail = df['on_trail'].values.astype(bool)
water = df['near_water'].values.astype(bool)
base  = np.where(road, 1,
        np.where(trail & (slope < 15), 2,
        np.where(trail | (slope < 25), 3, 4)))
df['custom_difficulty'] = np.where(water & ~road, np.minimum(base + 1, 5), base).astype(int)

# Знаковый уклон (+ вверх, - вниз)
df['signed_slope_deg'] = np.degrees(np.arctan2(
    df['elev_diff_m'].values,
    np.maximum(df['dist_m'].values, 0.1)
))

# Порядковый номер сегмента внутри трека
df['seg_idx'] = df.groupby('track_id').cumcount()

print(f'Сегментов: {len(df):,}  треков: {df["track_id"].nunique():,}')
print(f'on_trail: {df["on_trail"].mean():.1%}  on_road: {df["on_road"].mean():.1%}')
print(f'near_water: {df["near_water"].mean():.1%}  off_trail: {df["off_trail"].mean():.1%}')

## 2. Восстановление треков как последовательностей

In [ ]:
# Признаки для LSTM (point-level features)
POINT_FEATURES = [
    # Рельеф (как в HikingTTE)
    'signed_slope_deg',  # уклон со знаком
    'elev_diff_m',       # перепад высоты
    'dist_m',            # длина сегмента
    # GIS-фичи (наш вклад сверх HikingTTE)
    'on_trail',
    'on_road',
    'off_trail',
    'near_water',
    'log_dist_trail',
    'log_dist_road',
    'custom_difficulty',
]

# Атрибуты трека (как в HikingTTE)
ATTR_FEATURES = [
    'elemax', 'elemin', 'D_plus', 'D_minus', 'disttotal',
    # vhat_{slope}: оценка индивидуальной способности на 7 уклонах
    'vhat_n30', 'vhat_n20', 'vhat_n10', 'vhat_0', 'vhat_p10', 'vhat_p20', 'vhat_p30',
]

SPLIT_RATIO = 0.3  # первые 30% трека -> оценка способности
MIN_SEGS    = 20   # минимум сегментов в треке

def campbell_speed_kmh(slope_deg):
    """Упрощённая Campbell-функция -> скорость км/ч."""
    s = np.asarray(slope_deg, dtype=float)
    a, b, c, d, e = -1.4579, 22.0787, 76.3271, 0.0525, 3.2002e-4
    speed_mps = c * (1.0 / (np.pi * b * (1.0 + ((s + a) / b) ** 2))) + d + e * s
    return np.maximum(speed_mps, 0.01) * 3.6

def estimate_ability_attrs(front_df):
    """Оценка индивидуальной ходовой способности по первым X% трека."""
    valid = front_df[front_df['speed_kmh'].between(0.3, 12.0)].copy()
    if len(valid) < 5:
        mult = 1.0
    else:
        base_speed = campbell_speed_kmh(valid['signed_slope_deg'].values)
        ratio = valid['speed_kmh'].values / np.maximum(base_speed, 0.1)
        mult = float(np.median(np.clip(ratio, 0.2, 4.0)))

    fixed_slopes = [-30, -20, -10, 0, 10, 20, 30]
    vhats = campbell_speed_kmh(np.array(fixed_slopes, dtype=float)) * mult
    keys  = ['vhat_n30','vhat_n20','vhat_n10','vhat_0','vhat_p10','vhat_p20','vhat_p30']
    return dict(zip(keys, vhats))

def build_track_samples(df, split_ratio=SPLIT_RATIO, min_segs=MIN_SEGS):
    samples = []
    for tid, gdf in df.groupby('track_id'):
        gdf = gdf.sort_values('seg_idx').reset_index(drop=True)
        n = len(gdf)
        if n < min_segs:
            continue

        split_idx = max(3, int(n * split_ratio))
        front = gdf.iloc[:split_idx]
        back  = gdf.iloc[split_idx:]
        if len(back) < 5:
            continue

        # target: время остатка маршрута (ч)
        y_entire = float(back['dt_s'].sum()) / 3600.0
        if y_entire <= 0.01:
            continue

        # атрибуты трека
        ele = gdf['elev_diff_m'].cumsum().values
        attr = {
            'elemax':    float(ele.max()),
            'elemin':    float(ele.min()),
            'D_plus':    float(np.maximum(gdf['elev_diff_m'].values, 0).sum()),
            'D_minus':   float(np.maximum(-gdf['elev_diff_m'].values, 0).sum()),
            'disttotal': float(gdf['dist_m'].sum()),
        }
        attr.update(estimate_ability_attrs(front))

        # point features для back-части
        X_seq     = back[POINT_FEATURES].fillna(0).values.astype(np.float32)
        y_local   = (back['dt_s'].values / 3600.0).astype(np.float32)  # ч на сегмент
        attr_vec  = np.array([attr[k] for k in ATTR_FEATURES], dtype=np.float32)

        samples.append({
            'track_id': tid,
            'X_seq':    X_seq,
            'attr':     attr_vec,
            'y_entire': np.float32(y_entire),
            'y_local':  y_local,
        })
    return samples

print('Строим треки-последовательности...')
t0 = time.time()
all_samples = build_track_samples(df)
print(f'Готово за {time.time()-t0:.1f}с')
print(f'Треков-сэмплов: {len(all_samples):,}')

# Длины треков
lens = [len(s['X_seq']) for s in all_samples]
print(f'Длина back-части: min={min(lens)} median={int(np.median(lens))} max={max(lens)} p95={int(np.percentile(lens,95))}')

## 3. Нормализация + train/val/test split

In [ ]:
# -- Canonical split (shared: NB-05, NB-05b) ------------------------------
# NB-05 создаёт canonical_split.pkl, NB-06 загружает (или создаёт если запущен первым)
_split_path = CACHE_DIR / 'canonical_split.pkl'
if _split_path.exists():
    _sp = pickle.load(open(_split_path, 'rb'))
    te_ids, val_ids, tr_ids = _sp['te_ids'], _sp['val_ids'], _sp['tr_ids']
    print(f'Loaded canonical split: train={len(tr_ids)} val={len(val_ids)} test={len(te_ids)} треков')
else:
    _uids = sorted(df['track_id'].unique().tolist())
    _rng  = np.random.default_rng(42)
    _idx  = np.arange(len(_uids))
    _rng.shuffle(_idx)
    _n_te  = max(1, int(len(_idx) * 0.15))
    _n_val = max(1, int(len(_idx) * 0.15))
    te_ids  = {_uids[i] for i in _idx[:_n_te]}
    val_ids = {_uids[i] for i in _idx[_n_te:_n_te+_n_val]}
    tr_ids  = {_uids[i] for i in _idx[_n_te+_n_val:]}
    pickle.dump({'tr_ids': tr_ids, 'val_ids': val_ids, 'te_ids': te_ids},
                open(_split_path, 'wb'))
    print(f'Created canonical split: train={len(tr_ids)} val={len(val_ids)} test={len(te_ids)} треков')

tr_samples  = [s for s in all_samples if s['track_id'] in tr_ids]
val_samples = [s for s in all_samples if s['track_id'] in val_ids]
te_samples  = [s for s in all_samples if s['track_id'] in te_ids]
print(f'Сэмплов: train={len(tr_samples):,}  val={len(val_samples):,}  test={len(te_samples):,}')

# Нормализация по train only
X_all_tr = np.concatenate([s['X_seq'] for s in tr_samples], axis=0)
A_all_tr = np.stack([s['attr'] for s in tr_samples])
x_scaler = StandardScaler().fit(X_all_tr)
a_scaler = StandardScaler().fit(A_all_tr)

def norm_sample(s):
    s = dict(s)
    s['X_seq'] = x_scaler.transform(s['X_seq']).astype(np.float32)
    s['attr']  = a_scaler.transform(s['attr'][None])[0].astype(np.float32)
    return s

tr_samples  = [norm_sample(s) for s in tr_samples]
val_samples = [norm_sample(s) for s in val_samples]
te_samples  = [norm_sample(s) for s in te_samples]
print('Нормализация применена.')


## 4. Dataset и DataLoader

In [ ]:
MAX_LEN = 256

class TrackDataset(Dataset):
    def __init__(self, samples, max_len=MAX_LEN):
        self.samples = samples
        self.max_len = max_len

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        X, y_l = s['X_seq'], s['y_local']
        if len(X) > self.max_len:
            X, y_l = X[:self.max_len], y_l[:self.max_len]
        # y_entire должен соответствовать тому, что видит LSTM
        y_entire = float(y_l.sum())
        return {
            'X_seq':    torch.tensor(X,        dtype=torch.float32),
            'attr':     torch.tensor(s['attr'], dtype=torch.float32),
            'y_entire': torch.tensor(y_entire,  dtype=torch.float32),
            'y_local':  torch.tensor(y_l,       dtype=torch.float32),
            'length':   torch.tensor(len(X),    dtype=torch.long),
        }

def collate_fn(batch):
    lengths  = torch.stack([b['length'] for b in batch])
    max_len  = int(lengths.max())
    n_feat   = batch[0]['X_seq'].shape[1]
    n_attr   = batch[0]['attr'].shape[0]
    B        = len(batch)

    X       = torch.zeros(B, max_len, n_feat)
    y_local = torch.zeros(B, max_len)
    mask    = torch.zeros(B, max_len)
    attr    = torch.zeros(B, n_attr)
    y_ent   = torch.zeros(B)

    for i, b in enumerate(batch):
        L = int(b['length'])
        X[i, :L]       = b['X_seq']
        y_local[i, :L] = b['y_local']
        mask[i, :L]    = 1.0
        attr[i]  = b['attr']
        y_ent[i] = b['y_entire']

    return {'X_seq': X, 'attr': attr, 'y_entire': y_ent,
            'y_local': y_local, 'mask': mask, 'lengths': lengths}

BATCH = 64
tr_loader  = DataLoader(TrackDataset(tr_samples),  batch_size=BATCH, shuffle=True,  collate_fn=collate_fn, num_workers=0)
val_loader = DataLoader(TrackDataset(val_samples), batch_size=BATCH, shuffle=False, collate_fn=collate_fn, num_workers=0)
te_loader  = DataLoader(TrackDataset(te_samples),  batch_size=BATCH, shuffle=False, collate_fn=collate_fn, num_workers=0)
print(f'Batches train={len(tr_loader)}  val={len(val_loader)}  test={len(te_loader)}')

## 5. Модель: LSTM + Attention + Multi-task

In [ ]:
class HikingTTEGIS(nn.Module):
    """
    HikingTTE-like модель расширенная GIS-признаками.
    Два выхода:
      - local_head:  скорость (км/ч) каждого сегмента -> для cost-surface
      - entire_head: суммарное время остатка (ч) -> для маршрутного TTE
    """
    def __init__(self, n_point, n_attr, loc_dim=32, hidden=128, n_layers=2, dropout=0.2):
        super().__init__()
        self.loc_proj = nn.Sequential(
            nn.Linear(n_point, loc_dim), nn.Tanh()
        )
        self.lstm = nn.LSTM(
            input_size=loc_dim + n_attr,
            hidden_size=hidden,
            num_layers=n_layers,
            batch_first=True,
            dropout=dropout if n_layers > 1 else 0.0
        )
        # local: предсказывает время сегмента (ч)
        self.local_head = nn.Sequential(
            nn.Linear(hidden, 64), nn.LeakyReLU(),
            nn.Linear(64, 32),    nn.LeakyReLU(),
            nn.Linear(32, 1)
        )
        # attention query для entire head
        self.attn_q = nn.Linear(n_attr, hidden)
        # entire: суммарное время остатка
        self.entire_head = nn.Sequential(
            nn.Linear(hidden + n_attr, hidden), nn.LeakyReLU(),
            nn.Linear(hidden, hidden // 2),      nn.LeakyReLU(),
            nn.Linear(hidden // 2, 1)
        )

    def forward(self, X_seq, attr, mask):
        B, T, _ = X_seq.shape
        loc = self.loc_proj(X_seq)
        attr_rep = attr.unsqueeze(1).expand(-1, T, -1)
        h, _ = self.lstm(torch.cat([loc, attr_rep], dim=-1))

        # local time per segment
        y_local = F.softplus(self.local_head(h).squeeze(-1))

        # attention over sequence
        q = self.attn_q(attr).unsqueeze(-1)
        scores  = torch.bmm(h, q).squeeze(-1)
        scores  = scores.masked_fill(mask == 0, -1e9)
        weights = torch.softmax(scores, dim=1)
        context = (h * weights.unsqueeze(-1)).sum(dim=1)

        y_entire = F.softplus(self.entire_head(torch.cat([context, attr], dim=-1)).squeeze(-1))
        return y_entire, y_local, weights


def masked_mape(pred, true, mask=None, eps=1e-3):
    err = torch.abs((pred - true) / (true.abs() + eps))
    if mask is not None:
        return (err * mask).sum() / (mask.sum() + eps)
    return err.mean()


def multitask_loss(y_ent_p, y_ent, y_loc_p, y_loc, mask, alpha=0.5):
    l_ent = masked_mape(y_ent_p, y_ent)
    l_loc = masked_mape(y_loc_p, y_loc, mask)
    return alpha * l_loc + (1 - alpha) * l_ent, l_ent, l_loc


n_point = len(POINT_FEATURES)
n_attr  = len(ATTR_FEATURES)
model   = HikingTTEGIS(n_point, n_attr, loc_dim=32, hidden=128, n_layers=2, dropout=0.2).to(device)
total   = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Параметры: {total:,}')
print(f'Point features ({n_point}): {POINT_FEATURES}')
print(f'Attr features  ({n_attr}): {ATTR_FEATURES}')

## 6. Обучение

In [ ]:
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    preds, targets = [], []
    # для per-segment метрики (local head -> speed)
    seg_preds, seg_true = [], []

    for batch in loader:
        X   = batch['X_seq'].to(device)
        A   = batch['attr'].to(device)
        msk = batch['mask'].to(device)
        y_e = batch['y_entire']
        y_l = batch['y_local']

        y_ep, y_lp, _ = model(X, A, msk)

        preds.extend(y_ep.cpu().numpy())
        targets.extend(y_e.numpy())

        # local: переводим время (ч) -> скорость (км/ч) для сравнения с CatBoost
        # seg_dist_km нужен; используем batch маску и dist_m из X_seq (индекс 2)
        dist_km = X[:, :, 2].cpu().numpy() / 1000.0  # dist_m -> km (ненормализованный!)
        # Внимание: X уже нормализован -> dist нельзя использовать напрямую.
        # Сохраняем y_local (время ч) для MAPE по времени (как у авторов).
        mask_np = msk.cpu().numpy().astype(bool)
        seg_preds.extend(y_lp.cpu().numpy()[mask_np])
        seg_true.extend(y_l.numpy()[mask_np])

    p, t = np.array(preds), np.array(targets)
    mape_e = 100 * np.mean(np.abs((p - t) / np.maximum(t, 1e-3)))
    mae_e  = mean_absolute_error(t, p)

    sp, st = np.array(seg_preds), np.array(seg_true)
    mape_l = 100 * np.mean(np.abs((sp - st) / np.maximum(st, 1e-4)))

    return {'MAPE_entire': mape_e, 'MAE_entire_h': mae_e, 'MAPE_local': mape_l}


optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=10, factor=0.5)

EPOCHS   = 200
PATIENCE = 30
ALPHA    = 0.5

best_val_mape = float('inf')
best_state    = None
no_improve    = 0
history       = []

print(f'Обучение: {EPOCHS} эпох, patience={PATIENCE}')
t0 = time.time()

for epoch in range(1, EPOCHS + 1):
    model.train()
    losses = []
    for batch in tr_loader:
        X   = batch['X_seq'].to(device)
        A   = batch['attr'].to(device)
        msk = batch['mask'].to(device)
        y_e = batch['y_entire'].to(device)
        y_l = batch['y_local'].to(device)

        optimizer.zero_grad()
        y_ep, y_lp, _ = model(X, A, msk)
        loss, _, _ = multitask_loss(y_ep, y_e, y_lp, y_l, msk, ALPHA)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        losses.append(loss.item())

    val_m = evaluate(model, val_loader)
    val_mape = val_m['MAPE_entire']
    scheduler.step(val_mape)

    history.append({'epoch': epoch, 'train_loss': np.mean(losses), **val_m})

    if val_mape < best_val_mape:
        best_val_mape = val_mape
        best_state    = copy.deepcopy(model.state_dict())
        no_improve    = 0
    else:
        no_improve += 1

    if epoch % 10 == 0:
        print(f'Epoch {epoch:03d} | loss={np.mean(losses):.4f} | '
              f'val_MAPE_ent={val_mape:.2f}% | val_MAE={val_m["MAE_entire_h"]:.4f}h | '
              f'lr={optimizer.param_groups[0]["lr"]:.2e}')

    if no_improve >= PATIENCE:
        print(f'Early stopping at epoch {epoch}')
        break

model.load_state_dict(best_state)
print(f'\nГотово за {(time.time()-t0)/60:.1f} мин')
print(f'Best val MAPE (entire): {best_val_mape:.2f}%')

## 7. Оценка и сравнение с CatBoost

In [ ]:
from catboost import CatBoostRegressor

def tobler_kmh(slope_deg):
    s = np.asarray(slope_deg, dtype=float)
    return 6 * np.exp(-3.5 * np.abs(np.tan(np.radians(s)) + 0.05))

CB03_FEAT = ['log_dist', 'slope_deg', 'slope_sq', 'elev_diff_m',
             'going_up', 'slope_x_up', 'difficulty']
CB03_CAT  = ['going_up', 'difficulty']

CB05_FEAT = ['signed_slope_deg', 'elev_diff_m', 'on_trail', 'on_road',
             'off_trail', 'near_water', 'log_dist_trail', 'log_dist_road',
             'custom_difficulty']
CB05_CAT  = ['on_trail', 'on_road', 'off_trail', 'near_water', 'custom_difficulty']

def prep(df_slice, features, cat_cols):
    X = df_slice[features].fillna(0).copy()
    for c in cat_cols:
        X[c] = X[c].astype(int)
    return X

cb03 = CatBoostRegressor(); cb03.load_model(str(MODELS_DIR / 'travel_time_hikr.cbm'))
cb05 = CatBoostRegressor(); cb05.load_model(str(MODELS_DIR / 'travel_time_opt.cbm'))

# weights_only=False т.к. чекпоинт содержит numpy-массивы (scalers)
_ckpt = torch.load(str(MODELS_DIR / 'hiking_tte_gis.pt'), map_location='cpu', weights_only=False)
te_m  = _ckpt['test_metrics']

# Загружаем history из чекпоинта как fallback (если не обучали в этой сессии)
if not ('history' in dir() and isinstance(history, list) and len(history) > 0):
    history = _ckpt.get('history', [])

te_tids   = {s['track_id'] for s in te_samples}
df_te_seg = df[df['track_id'].isin(te_tids)]
y_spd     = df_te_seg['speed_kmh'].values

yp_cb03 = cb03.predict(prep(df_te_seg, CB03_FEAT, CB03_CAT))
yp_cb05 = cb05.predict(prep(df_te_seg, CB05_FEAT, CB05_CAT))

r2_cb03  = r2_score(y_spd, yp_cb03)
mae_cb03 = mean_absolute_error(y_spd, yp_cb03)
r2_cb05  = r2_score(y_spd, yp_cb05)
mae_cb05 = mean_absolute_error(y_spd, yp_cb05)

def tte_mape(samples, df, predict_speed_fn, speed_min=0.3):
    p_list, a_list = [], []
    for s in samples:
        tid   = s['track_id']
        tdf   = df[df['track_id'] == tid].sort_values('seg_idx').reset_index(drop=True)
        n     = len(tdf)
        split = max(3, int(n * SPLIT_RATIO))
        back  = tdf.iloc[split:]
        if len(back) < 5:
            continue
        spd    = np.maximum(predict_speed_fn(back), speed_min)
        time_h = back['dist_m'].values / (spd / 3.6) / 3600
        p_list.append(time_h.sum())
        a_list.append(float(s['y_entire']))
    p, a = np.array(p_list), np.array(a_list)
    return 100 * np.mean(np.abs((p - a) / np.maximum(a, 1e-3)))

mape_tobler   = tte_mape(te_samples, df,
                         lambda d: tobler_kmh(d['signed_slope_deg'].values))
mape_cb03_tte = tte_mape(te_samples, df,
                         lambda d: cb03.predict(prep(d, CB03_FEAT, CB03_CAT)))
mape_cb05_tte = tte_mape(te_samples, df,
                         lambda d: cb05.predict(prep(d, CB05_FEAT, CB05_CAT)))

print('=== TEST RESULTS (LSTM) ===')
print(f'MAPE entire path: {te_m["MAPE_entire"]:.2f}%')
print(f'MAE entire path:  {te_m["MAE_entire_h"]:.4f} ч')
print(f'MAPE local segs:  {te_m["MAPE_local"]:.2f}%')

print()
print('=== Сравнение ===')
rows = [
    ('Baseline Tobler',              '-',               '-',               f'{mape_tobler:.2f}%'),
    ('HikingTTE (Asako et al. X=0)', '-',               '-',               '15.62%'),
    ('NB-03 CatBoost (slope+SAC)',   f'{r2_cb03:.3f}',  f'{mae_cb03:.3f}', f'{mape_cb03_tte:.2f}%'),
    ('NB-05 CatBoost + GIS',         f'{r2_cb05:.3f}',  f'{mae_cb05:.3f}', f'{mape_cb05_tte:.2f}%'),
    ('NB-06 LSTM + GIS (наш)',       '-',               '-',               f'{te_m["MAPE_entire"]:.2f}%'),
]
print(f'{"Модель":<40} {"R^2":>6}  {"MAE км/ч":>9}  {"MAPE TTE":>9}')
print('-' * 70)
for name, r2_str, mae_str, mape_str in rows:
    print(f'{name:<40} {r2_str:>6}  {mae_str:>9}  {mape_str:>9}')

if isinstance(history, list) and len(history) > 0:
    hist_df = pd.DataFrame(history)
    x_axis  = hist_df['epoch'] if 'epoch' in hist_df.columns else hist_df.index
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    if 'train_loss' in hist_df.columns:
        axes[0].plot(x_axis, hist_df['train_loss'], label='train loss')
    if 'MAPE_entire' in hist_df.columns:
        axes[0].plot(x_axis, hist_df['MAPE_entire'] / 100, label='val MAPE (entire)')
    axes[0].set_xlabel('Epoch'); axes[0].set_title('Training history'); axes[0].legend()
    if 'MAE_entire_h' in hist_df.columns:
        axes[1].plot(x_axis, hist_df['MAE_entire_h'])
    axes[1].set_xlabel('Epoch'); axes[1].set_title('Val MAE (ч)')
    plt.tight_layout()
    plt.savefig('figures/hiking_tte_gis_training.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'График сохранён: figures/hiking_tte_gis_training.png ({len(history)} эпох)')
else:
    print('history недоступна в этой сессии (модель загружена из чекпоинта)')


## 8. Сохранение модели

In [ ]:
torch.save({
    'model_state_dict': model.state_dict(),
    'model_config': {
        'n_point':  n_point,
        'n_attr':   n_attr,
        'loc_dim':  32,
        'hidden':   128,
        'n_layers': 2,
        'dropout':  0.2,
    },
    'point_features': POINT_FEATURES,
    'attr_features':  ATTR_FEATURES,
    'x_scaler_mean':  x_scaler.mean_,
    'x_scaler_std':   x_scaler.scale_,
    'a_scaler_mean':  a_scaler.mean_,
    'a_scaler_std':   a_scaler.scale_,
    'test_metrics':   te_m,
    'history':        history,
    'split_ratio':    SPLIT_RATIO,
}, 'models/hiking_tte_gis.pt')

print('Сохранено: models/hiking_tte_gis.pt')
print(f'История обучения: {len(history)} эпох сохранено в чекпоинт')
print()
print('Для инференса per-segment speed (для SAR cost-surface):')
print('  1. Загрузить модель, применить x_scaler к point features')
print('  2. Прогнать через LSTM -> y_local (время сегмента, ч)')
print('  3. speed_kmh = dist_m / 1000 / y_local_h')
print()
print('Отличие от HikingTTE (для диссертации):')
print('  Добавлены GIS-признаки: on_trail, on_road, near_water,')
print('  log_dist_trail, log_dist_road, custom_difficulty')
print('  - то, что авторы называют ограничением своего подхода.')
